# LongFlow P1 -- capture v2, the real 20K-step training run

Runtime: **A100 GPU** (this is the real spend, wall-clock matters more
than $/hr -- Josh's call, 2026-08-16). Expect ~1-2h total (training +
held-out decode across 4 checkpoints).

The gate check (5K steps) and the shaky-voice ablation both pointed the
same direction: the wobble everyone heard is generic undertraining, not
capture v2's data or the clean/noised mixing (both were directly ruled
out -- see NOTES.md "Shaky-voice ablation" entries). July's 20K-step
head on the OLD cache was clean; this run tests whether 20K steps on the
NEW cache is too. Per July's own E3 finding, 20K was ALREADY the sweet
spot on a similarly-sized cache (478K frames) -- 80K steps overfit and
got WORSE on held-out data. Capture v2 is comparably sized (510K frames),
so the target here is 20K, not higher -- going past that has already
been shown to hurt, not help, at this data scale.

What's different from every quick check so far, per the project's own
"process rules now in force" (2026-08-11) that the gate/ablation
notebooks didn't implement:
- **A real held-out split** (5 scripts per word bin, 25 total, stratified
  by length, never trained on) -- not just training-set listening.
- **Intermediate checkpoints every 5K steps** (5K/10K/15K/20K) -- lets us
  verify the 20K-is-the-sweet-spot claim empirically on THIS cache
  instead of assuming it transfers from v1's.
- **The proven LR schedule** (2e-4 -> 2e-5 cosine decay) and **EMA decay
  0.9999** (the documented recipe default, appropriate for a run this
  long -- the gate/ablation checks used a faster EMA deliberately sized
  for their much shorter runs).

Colab generates; Mac analyzes -- this notebook renders held-out audio and
a manifest, zips, downloads. Run `score_train20k_v2.py` locally afterward
for WER/speaker-sim via `src/eval/metrics.py` (same pattern as every
gate-night scorer).

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "Train v2 20K v1.0 (2026-08-17): real training run + held-out eval"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
from pathlib import Path
import soundfile as sf
import numpy as np

CACHE_V2_DIR = "/content/drive/MyDrive/longflow_p1_cache_v2"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists(CACHE_V2_DIR), f"{CACHE_V2_DIR} not found -- capture v2 must be run first"

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed -- check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import PairData, save_checkpoint, load_checkpoint, train, sample_latents

def decode_latents(z, chunk_frames=225):  # 225 frames = 30s @ 7.5Hz
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt {tuple(fn(chunk).shape)} failed: {repr(e)[:150]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed -- paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

print("READY")

In [ ]:
# ===== Stratified held-out split + training pool (5 per word bin, never trained on) =====
HELD_OUT_PER_BIN = 5

v2_files = sorted(glob.glob(f"{CACHE_V2_DIR}/*.pt"))
print(f"{len(v2_files)} total v2 scripts")

by_bin = {}
for f in v2_files:
    d = torch.load(f, weights_only=True)
    tw = d["meta"]["target_words"]
    by_bin.setdefault(tw, []).append(f)

held_out_by_bin = {tw: files[:HELD_OUT_PER_BIN] for tw, files in sorted(by_bin.items())}
held_out_files = [f for files in held_out_by_bin.values() for f in files]
train_files = [f for tw, files in sorted(by_bin.items()) for f in files[HELD_OUT_PER_BIN:]]
print(f"held-out: {len(held_out_files)} scripts across {len(held_out_by_bin)} bins")
print(f"train: {len(train_files)} scripts")

json.dump(
    {"held_out": [Path(f).name for f in held_out_files]},
    open(f"{CKPT_DIR}/v2_20k_held_out_manifest.json", "w"), indent=2,
)

def pairs_from_files(files):
    hiddens, latents = [], []
    for f in files:
        utt = load_utterance(f)
        hiddens.append(utt.hidden.float())
        latents.append(utt.latent.float())
    hidden = torch.cat(hiddens)
    latent = torch.cat(latents)
    mean = latent.mean(dim=0)
    std = latent.std(dim=0).clamp_min(1e-4)
    return PairData(hidden=hidden, latent=(latent - mean) / std, mean=mean, std=std)

train_data = pairs_from_files(train_files)
print(f"training pool: {train_data.hidden.shape[0]} frames  d_model={train_data.d_model}  d_latent={train_data.d_latent}")

In [ ]:
# ===== Train 20K steps, checkpoints every 5K, proven LR schedule + EMA =====
CKPT_TAG = "v2_20k"

def ckpt_path(step):
    return f"{CKPT_DIR}/{CKPT_TAG}_step{step}.pt"

head = FlowHead(FlowHeadConfig(d_model=train_data.d_model, d_latent=train_data.d_latent))
print(f"head params: {head.param_count()/1e6:.2f}M")

t0 = time.time()
out = train(
    head, train_data, steps=20000, batch_size=1024, lr=2e-4, lr_final=2e-5,
    ema_decay=0.9999, device="cuda", log_every=1000,
    checkpoint_every=5000, checkpoint_path_fn=ckpt_path,
)
print(f"training done in {(time.time()-t0)/60:.1f} min")
print(f"checkpoints saved: {[ckpt_path(s) for s in (5000, 10000, 15000, 20000)]}")

In [ ]:
# ===== Held-out decode: teacher (once) + flow4 per checkpoint =====
# Full 25-utterance eval only at the final checkpoint (the real number);
# intermediate checkpoints get a 2-per-bin subset -- enough to trace the
# overfitting curve (July's E3 methodology) without paying full cost 4x.
EVAL_DIR = "/content/train20k_eval"
os.makedirs(EVAL_DIR, exist_ok=True)
SUBSET_PER_BIN = 2
FULL_EVAL_STEPS = {20000}

manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}

for f in held_out_files:
    utt = load_utterance(f)
    wav = decode_latents(utt.latent.float())
    audio_name = f"{utt.utt_id}_teacher.wav"
    sf.write(f"{EVAL_DIR}/{audio_name}", wav, 24000)
    manifest["teacher"][utt.utt_id] = {
        "audio": audio_name, "text": utt.text, "target_words": utt.meta["target_words"],
    }
print(f"{len(manifest['teacher'])} teacher references rendered")

subset_by_bin = {tw: files[:SUBSET_PER_BIN] for tw, files in held_out_by_bin.items()}
subset_files = [f for files in subset_by_bin.values() for f in files]

for step in (5000, 10000, 15000, 20000):
    print(f"=== evaluating checkpoint step {step} ===")
    head_ckpt, mean_ckpt, std_ckpt = load_checkpoint(ckpt_path(step))
    head_ckpt = head_ckpt.to("cuda")
    eval_files = held_out_files if step in FULL_EVAL_STEPS else subset_files
    entries = []
    for f in eval_files:
        utt = load_utterance(f)
        z = sample_latents(head_ckpt, utt.hidden.float(), mean_ckpt, std_ckpt, nfe=4)
        wav = decode_latents(z)
        audio_name = f"{utt.utt_id}_step{step}_flow4.wav"
        sf.write(f"{EVAL_DIR}/{audio_name}", wav, 24000)
        entries.append({
            "utt_id": utt.utt_id, "audio": audio_name,
            "teacher_audio": manifest["teacher"][utt.utt_id]["audio"],
            "text": utt.text, "target_words": utt.meta["target_words"],
        })
    manifest["checkpoints"][step] = entries
    del head_ckpt
    torch.cuda.empty_cache()
    print(f"  {len(entries)} utterances rendered")

json.dump(manifest, open(f"{EVAL_DIR}/manifest.json", "w"), indent=2)
print("DONE -- manifest + audio ready to bundle")

In [ ]:
# ===== Bundle for download (Mac scores it: score_train20k_v2.py) =====
import zipfile
with zipfile.ZipFile("/content/train20k_v2_eval.zip", "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
from google.colab import files as colab_files
colab_files.download("/content/train20k_v2_eval.zip")